In [0]:
CATALOG = "opensky_lakehouse"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.flight_states"

GOLD_AIR_TRAFFIC_BY_MINUTE = f"{CATALOG}.{GOLD_SCHEMA}.air_traffic_by_minute"
GOLD_AIR_TRAFFIC_BY_COUNTRY = f"{CATALOG}.{GOLD_SCHEMA}.air_traffic_by_country"
GOLD_AIRCRAFT_LATEST_POSITION = f"{CATALOG}.{GOLD_SCHEMA}.aircraft_latest_position"
GOLD_ALTITUDE_DISTRIBUTION = f"{CATALOG}.{GOLD_SCHEMA}.altitude_distribution"
GOLD_FLIGHT_ACTIVITY_SUMMARY = f"{CATALOG}.{GOLD_SCHEMA}.flight_activity_summary"

In [0]:
# Se crea una tabla Gold si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

In [0]:
# Comprobación de datos en Silver
silver_count = spark.table(SILVER_TABLE).count()

print(f"Registros disponibles en Silver: {silver_count}")

display(spark.table(SILVER_TABLE).limit(20))

# 1. Tabla Gold: tráfico aéreo por minuto
### Esta tabla responde a:
 - ¿Cuántos aviones hay activos por minuto?
 - ¿Cuál es la velocidad media?
 - ¿Cuál es la altitud media?
 - ¿Cuántos están en tierra o en vuelo?

In [0]:
# Gold: air_traffic_by_minute
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_AIR_TRAFFIC_BY_MINUTE}
USING DELTA
AS
WITH base AS (
    SELECT
        event_time,
        icao24,
        origin_country,
        velocity_kmh,
        baro_altitude_m,
        geo_altitude_m,
        on_ground
    FROM {SILVER_TABLE}
    WHERE event_time IS NOT NULL
      AND icao24 IS NOT NULL
),

aggregated AS (
    SELECT
        window(event_time, '1 minute') AS time_window,

        COUNT(*) AS total_records,
        COUNT(DISTINCT icao24) AS active_aircraft,
        COUNT(DISTINCT origin_country) AS unique_origin_countries,

        ROUND(AVG(velocity_kmh), 2) AS avg_velocity_kmh,
        ROUND(MIN(velocity_kmh), 2) AS min_velocity_kmh,
        ROUND(MAX(velocity_kmh), 2) AS max_velocity_kmh,

        ROUND(AVG(baro_altitude_m), 2) AS avg_baro_altitude_m,
        ROUND(MIN(baro_altitude_m), 2) AS min_baro_altitude_m,
        ROUND(MAX(baro_altitude_m), 2) AS max_baro_altitude_m,

        SUM(CASE WHEN on_ground = true THEN 1 ELSE 0 END) AS records_on_ground,
        SUM(CASE WHEN on_ground = false THEN 1 ELSE 0 END) AS records_in_air
    FROM base
    GROUP BY window(event_time, '1 minute')
)

SELECT
    time_window.start AS window_start,
    time_window.end AS window_end,
    DATE(time_window.start) AS event_date,

    total_records,
    active_aircraft,
    unique_origin_countries,

    avg_velocity_kmh,
    min_velocity_kmh,
    max_velocity_kmh,

    avg_baro_altitude_m,
    min_baro_altitude_m,
    max_baro_altitude_m,

    records_on_ground,
    records_in_air,

    current_timestamp() AS gold_updated_at
FROM aggregated
ORDER BY window_start
""")

display(spark.table(GOLD_AIR_TRAFFIC_BY_MINUTE))

# 2. Tabla Gold: tráfico por país
### Esta tabla responde a:
 - ¿Qué países de origen aparecen más?
 - ¿Qué países tienen más aviones activos?
 - ¿Cuál es la velocidad o altitud media por país?

In [0]:
# Gold: air_traffic_by_country

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_AIR_TRAFFIC_BY_COUNTRY}
USING DELTA
AS
SELECT
    event_date,

    COALESCE(origin_country, 'Unknown') AS origin_country,

    COUNT(*) AS total_records,
    COUNT(DISTINCT icao24) AS unique_aircraft,

    ROUND(AVG(velocity_kmh), 2) AS avg_velocity_kmh,
    ROUND(MIN(velocity_kmh), 2) AS min_velocity_kmh,
    ROUND(MAX(velocity_kmh), 2) AS max_velocity_kmh,

    ROUND(AVG(baro_altitude_m), 2) AS avg_baro_altitude_m,
    ROUND(MIN(baro_altitude_m), 2) AS min_baro_altitude_m,
    ROUND(MAX(baro_altitude_m), 2) AS max_baro_altitude_m,

    SUM(CASE WHEN on_ground = true THEN 1 ELSE 0 END) AS records_on_ground,
    SUM(CASE WHEN on_ground = false THEN 1 ELSE 0 END) AS records_in_air,

    MIN(event_time) AS first_seen,
    MAX(event_time) AS last_seen,

    current_timestamp() AS gold_updated_at
FROM {SILVER_TABLE}
WHERE event_date IS NOT NULL
GROUP BY
    event_date,
    COALESCE(origin_country, 'Unknown')
ORDER BY
    event_date DESC,
    unique_aircraft DESC
""")
display(spark.table(GOLD_AIR_TRAFFIC_BY_COUNTRY))

# 3. Tabla Gold: última posición conocida de cada avión



In [0]:
# Gold: aircraft_latest_position

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_AIRCRAFT_LATEST_POSITION}
USING DELTA
AS
WITH ranked AS (
    SELECT
        event_time,
        retrieved_at,

        icao24,
        callsign,
        origin_country,

        longitude,
        latitude,
        baro_altitude_m,
        geo_altitude_m,

        on_ground,
        velocity_kmh,
        true_track,
        vertical_rate,
        squawk,
        position_source,

        source_file,
        raw_hash,
        silver_ingestion_timestamp,

        ROW_NUMBER() OVER (
            PARTITION BY icao24
            ORDER BY event_time DESC, silver_ingestion_timestamp DESC
        ) AS rn
    FROM {SILVER_TABLE}
    WHERE icao24 IS NOT NULL
      AND event_time IS NOT NULL
      AND longitude IS NOT NULL
      AND latitude IS NOT NULL
)

SELECT
    event_time AS last_event_time,
    retrieved_at,

    icao24,
    callsign,
    origin_country,

    longitude,
    latitude,
    baro_altitude_m,
    geo_altitude_m,

    on_ground,
    velocity_kmh,
    true_track,
    vertical_rate,
    squawk,
    position_source,

    source_file,
    raw_hash,

    current_timestamp() AS gold_updated_at
FROM ranked
WHERE rn = 1
ORDER BY last_event_time DESC
""")

display(spark.table(GOLD_AIRCRAFT_LATEST_POSITION).limit(50))

# 4. Tabla Gold: distribución de altitudes
### Esta tabla responde a:
 - ¿A qué altitudes vuelan los aviones detectados?
 - ¿Cúantos estan a baja, media o alta altitud?



In [0]:
# Gold: altitude_distribution
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_ALTITUDE_DISTRIBUTION}
USING DELTA
AS
WITH base AS (
    SELECT
        event_date,
        icao24,
        velocity_kmh,
        baro_altitude_m,

        CASE
            WHEN baro_altitude_m IS NULL THEN 'Unknown'
            WHEN baro_altitude_m < 0 THEN '< 0 m'
            WHEN baro_altitude_m < 1000 THEN '0 - 1,000 m'
            WHEN baro_altitude_m < 3000 THEN '1,000 - 3,000 m'
            WHEN baro_altitude_m < 6000 THEN '3,000 - 6,000 m'
            WHEN baro_altitude_m < 9000 THEN '6,000 - 9,000 m'
            WHEN baro_altitude_m < 12000 THEN '9,000 - 12,000 m'
            ELSE '12,000+ m'
        END AS altitude_band,

        CASE
            WHEN baro_altitude_m IS NULL THEN 99
            WHEN baro_altitude_m < 0 THEN 0
            WHEN baro_altitude_m < 1000 THEN 1
            WHEN baro_altitude_m < 3000 THEN 2
            WHEN baro_altitude_m < 6000 THEN 3
            WHEN baro_altitude_m < 9000 THEN 4
            WHEN baro_altitude_m < 12000 THEN 5
            ELSE 6
        END AS altitude_band_order

    FROM {SILVER_TABLE}
    WHERE event_date IS NOT NULL
)

SELECT
    event_date,
    altitude_band,
    altitude_band_order,

    COUNT(*) AS total_records,
    COUNT(DISTINCT icao24) AS unique_aircraft,

    ROUND(AVG(velocity_kmh), 2) AS avg_velocity_kmh,
    ROUND(MIN(baro_altitude_m), 2) AS min_baro_altitude_m,
    ROUND(MAX(baro_altitude_m), 2) AS max_baro_altitude_m,

    current_timestamp() AS gold_updated_at
FROM base
GROUP BY
    event_date,
    altitude_band,
    altitude_band_order
ORDER BY
    event_date DESC,
    altitude_band_order
""")

display(spark.table(GOLD_ALTITUDE_DISTRIBUTION))

# 5. Tabla Gold: resumen general de la actividad


In [0]:
# Gold: flight_activity_summary

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_FLIGHT_ACTIVITY_SUMMARY}
USING DELTA
AS
SELECT
    event_date,

    COUNT(*) AS total_records,
    COUNT(DISTINCT icao24) AS unique_aircraft,
    COUNT(DISTINCT origin_country) AS unique_origin_countries,
    COUNT(DISTINCT source_file) AS source_files_processed,

    MIN(event_time) AS first_event_time,
    MAX(event_time) AS last_event_time,

    ROUND(AVG(velocity_kmh), 2) AS avg_velocity_kmh,
    ROUND(MIN(velocity_kmh), 2) AS min_velocity_kmh,
    ROUND(MAX(velocity_kmh), 2) AS max_velocity_kmh,

    ROUND(AVG(baro_altitude_m), 2) AS avg_baro_altitude_m,
    ROUND(MIN(baro_altitude_m), 2) AS min_baro_altitude_m,
    ROUND(MAX(baro_altitude_m), 2) AS max_baro_altitude_m,

    SUM(CASE WHEN on_ground = true THEN 1 ELSE 0 END) AS records_on_ground,
    SUM(CASE WHEN on_ground = false THEN 1 ELSE 0 END) AS records_in_air,

    ROUND(
        100.0 * SUM(CASE WHEN on_ground = true THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_records_on_ground,

    ROUND(
        100.0 * SUM(CASE WHEN on_ground = false THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_records_in_air,

    current_timestamp() AS gold_updated_at
FROM {SILVER_TABLE}
WHERE event_date IS NOT NULL
GROUP BY event_date
ORDER BY event_date DESC
""")

display(spark.table(GOLD_FLIGHT_ACTIVITY_SUMMARY))